# Inspect a saved World and build a graph Scenario

Load any `world.json` artifact and walk every inspection surface. Then derive a
graph-mode `Scenario` from it using `world_to_graph`.

**Before you click "Run All":** This notebook targets an *already-saved* world so
it never triggers an LLM call. The demo uses the offline fixture produced by
`example_llm_world_offline.py` (6-item fashion catalog, no OpenAI key required).

See `notebooks/01-openai_world_builder.ipynb` (quickstart 3-node chain) and
`notebooks/03-inspect_scenario.ipynb` (full scenario inspection with `nodes_df` / `edges_df`).

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from dataclasses import replace
from pathlib import Path

while not (Path.cwd() / 'pyproject.toml').exists():
    os.chdir('..')

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

from src.llm.world_builder import World

## Load a saved World

Point `WORLD_PATH` at any `world.json` produced by `WorldBuilder.build()` +
`World.to_json()` or the committed offline fixture.

In [ ]:
WORLD_PATH = Path('notebooks/fixtures/world_fashion_retail_offline.json')

world = World.from_json(WORLD_PATH)
print(f'Loaded world: {len(world.catalog)} catalog items, '
      f'{len(world.store_templates)} store templates')

### Meta — provenance

Records who built this world: archetype, item count, model name, builder
version, and build timestamp.

In [ ]:
world.meta_df()

### Market — demand and seasonality parameters

In [ ]:
world.market_df()

### Store templates — retained for world authoring

Store templates survive in the `World` artifact as authoring inputs even after
the graph migration. `world_to_graph` (below) synthesises graph topology from
them, so the LLM-authored capacity / balance / region information is preserved.

In [ ]:
world.store_templates_df().set_index('id')

### Catalog — items for sale

One row per SKU. `freshness_alpha` and `freshness_decay` govern the hype-curve
multiplier applied inside `DemandSinkNode.demand_target`. `stage_change_probs`
controls lifecycle-stage transition rates.

In [ ]:
world.catalog_df()

## Synthesise a graph topology with `world_to_graph`

`world_to_graph(world, *, sink_density)` converts the `store_templates` dict
into a set of `(nodes, edges)` ready for a graph-mode `Scenario`. Each template
becomes a 3-tier sub-graph:

```
FactoryNode(s) → IntermediateNode (shop) → DemandSinkNode × K products
```

`sink_density` controls the fraction of catalog products each sink covers
(default 1.0 = one sink per catalog product per shop).

In [ ]:
from src.sim.runner import build_world, world_to_graph
from src.sim.scenario import (
    DisruptionParams,
    ItemLifecycleParams,
    MarketParams,
    Scenario,
)
from datetime import datetime

nodes, edges = world_to_graph(world, sink_density=1.0)
print(f'Generated {len(nodes)} nodes and {len(edges)} edges')
node_types = [type(ni.node).__name__ for ni in nodes]
from collections import Counter
print('Node type counts:', dict(Counter(node_types)))

In [ ]:
from src.sim.distributions import Constant, Normal

LIFECYCLE_STAGES = ['introduction', 'growth', 'maturity', 'decline', 'dead']

scenario = Scenario(
    catalog=world.catalog,
    market=world.market,
    disruption=DisruptionParams(
        event_prob=0.02, types=['natural_disaster'], regions=world.market.regions,
        severity=Constant(0.05), duration=Constant(2),
    ),
    item_lifecycle=ItemLifecycleParams(
        stages=LIFECYCLE_STAGES, init_stage='maturity',
        default_stage_change_probs={s: 0.0 for s in LIFECYCLE_STAGES},
    ),
    stores=[],
    nodes=nodes,
    edges=edges,
    n_steps=20,
    start_date=datetime(2024, 1, 1),
    world_seed=99,
)

print(f'Scenario: {len(scenario.nodes)} nodes, {len(scenario.edges)} edges')
print(f'is_graph: {scenario.is_graph}')

## Inspect the graph scenario

`Scenario.nodes_df()` and `Scenario.edges_df()` are the canonical inspection views
for graph-mode scenarios — analogous to `catalog_df()` and `market_df()`.

In [ ]:
scenario.nodes_df()

In [ ]:
scenario.edges_df()

## Run the scenario and inspect the run log

In [ ]:
from src.sim.runner import Runner

run_log = Runner(scenario).run()
print('Run log keys:', list(run_log.keys()))
print(f'ticks logged: {len(run_log["ticks"])}')

In [ ]:
import matplotlib.pyplot as plt

# Plot per-tick cash for all intermediate nodes
from src.sim.node import IntermediateNode

shop_ids = [ni.node.id for ni in scenario.nodes if isinstance(ni.node, IntermediateNode)]

cash_series = {sid: [] for sid in shop_ids}
for tick_log in run_log['ticks']:
    for sid in shop_ids:
        cash_series[sid].append(tick_log['node_cash'].get(sid, 0.0))

fig, ax = plt.subplots(figsize=(10, 4))
for sid, vals in cash_series.items():
    ax.plot(vals, label=sid)
ax.set_title('Shop cash over time')
ax.set_xlabel('tick')
ax.set_ylabel('cash (£)')
ax.legend()
plt.tight_layout()
plt.show()

## Edit a world template (advanced)

`dataclasses.replace` produces a mutated `World` without touching the original.
Save under a distinct name to preserve the original artifact.

In [ ]:
# Example: bump flagship capacity (if the template exists)
if 'flagship' in world.store_templates:
    world_b = replace(
        world,
        store_templates={
            **world.store_templates,
            'flagship': replace(world.store_templates['flagship'], capacity=800),
        },
    )
    print('flagship capacity (original):', world.store_templates['flagship'].capacity)
    print('flagship capacity (edited):  ', world_b.store_templates['flagship'].capacity)
else:
    print('No flagship template in this world fixture — skipping edit demo.')
    print('Available templates:', list(world.store_templates.keys()))